In [ ]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from anp_emulator import Emulator
from train_anp_emulator import build_tasks, split_tasks, resolve_profile_file

In [ ]:
RUN_DIR = Path('anp_training_runs/anp_all_profiles_20260319_082722')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if not RUN_DIR.exists():
    raise FileNotFoundError(f'Run directory not found: {RUN_DIR}')

emu = Emulator.from_run_dir(RUN_DIR, device=DEVICE)
print('Loaded:', RUN_DIR)
print('Available fields:', emu.available_fields())

Loaded: anp_training_runs/anp_all_profiles_20260319_082722
Available fields: ['gas_density', 'temperature', 'pressure', 'metallicity']


In [ ]:
args = SimpleNamespace(**dict(emu.args))
all_families = build_tasks(args)
train_families, val_families, test_families = split_tasks(
    all_families,
    train_frac=float(args.train_frac),
    val_frac=float(args.val_frac),
    seed=int(args.seed),
)

test_run_ids = sorted([f.run_id for f in test_families])
print(f'Test families: {len(test_families)}')
print('Test run IDs:', test_run_ids)

In [ ]:
target_fields = ['temperature', 'pressure', 'gas_density', 'metallicity']
available = set(emu.available_fields())
missing = [f for f in target_fields if f not in available]
if missing:
    raise ValueError(f'Model does not contain required fields: {missing}')

def load_truth_arrays(run_id: int, snapnum: int):
    fpath = resolve_profile_file(
        run_id,
        base_path=Path(args.profiles_base),
        suite=args.suite,
        sim_set=args.sim_set,
        snapnum=int(snapnum),
    )
    with np.load(fpath) as dat:
        m500c = dat['M500c'].astype(np.float32)
        r500c = dat['R500c'].astype(np.float32)
        radial_bins = dat['radial_bins'].astype(np.float32)
        y_true = np.stack([
            dat['temperature_array'].astype(np.float32),
            dat['pressure_array'].astype(np.float32),
            dat['gas_density_array'].astype(np.float32),
            dat['metallicity_array'].astype(np.float32),
        ], axis=-1)
    return m500c, r500c, radial_bins, y_true

theta_table = pd.read_csv(Path(args.param_csv))
param_has_run_id = 'run_id' in theta_table.columns
if param_has_run_id:
    theta_table = theta_table.set_index('run_id')

snap_eval = int(getattr(args, 'snapnum', 90))
metric_rows = []
diag_rows = []
profile_cache = []
skipped_runs = []

for run_id in test_run_ids:
    try:
        m500c, r500c, radial_bins, y_true = load_truth_arrays(run_id=run_id, snapnum=snap_eval)
    except (FileNotFoundError, OSError) as exc:
        skipped_runs.append(int(run_id))
        print(f'Skipping run_id={run_id}: missing/unreadable profile file ({exc})')
        continue

    try:
        if param_has_run_id:
            theta = theta_table.loc[int(run_id)].to_numpy(dtype=np.float32)
        else:
            theta = theta_table.iloc[int(run_id)].to_numpy(dtype=np.float32)
    except (KeyError, IndexError) as exc:
        skipped_runs.append(int(run_id))
        print(f'Skipping run_id={run_id}: missing theta row ({exc})')
        continue

    rr500 = (radial_bins[None, :] / np.maximum(r500c[:, None], 1e-8)).astype(np.float32)

    try:
        pred = emu.predict(
            theta=theta,
            M=m500c.astype(np.float32),
            r_bins=rr500,
            field=target_fields,
            snapnum=snap_eval,
            n_samples=20,
        )
    except TypeError:
        pred = emu.predict(
            theta=theta,
            M=m500c.astype(np.float32),
            r_bins=rr500,
            field=target_fields,
            n_samples=20,
        )

    y_pred = pred.mean.astype(np.float64)
    y_true = y_true.astype(np.float64)

    profile_cache.append({
        'run_id': int(run_id),
        'radial_bins': radial_bins.copy(),
        'y_true': y_true.copy(),
        'y_pred': y_pred.copy(),
        'field_names': list(target_fields),
    })

    abs_err = np.abs(y_pred - y_true)
    for j, fname in enumerate(target_fields):
        yt = y_true[:, :, j]
        yp = y_pred[:, :, j]
        ae = abs_err[:, :, j]
        rmse = float(np.sqrt(np.mean((yp - yt) ** 2)))
        mae = float(np.mean(ae))
        denom = np.maximum(np.abs(yt), np.nanpercentile(np.abs(yt), 1.0) + 1e-30)
        mape = float(100.0 * np.mean(ae / denom))
        metric_rows.append({
            'run_id': int(run_id),
            'field': fname,
            'rmse': rmse,
            'mae': mae,
            'mape_pct_clipped': mape,
        })
        diag_rows.append(pd.DataFrame({'field': fname, 'true': yt.ravel(), 'pred': yp.ravel()}))

if not metric_rows:
    raise RuntimeError('No valid test runs were loaded; all runs were skipped.')

metrics_df = pd.DataFrame(metric_rows)
diag_df = pd.concat(diag_rows, ignore_index=True)
display(metrics_df.head())
if skipped_runs:
    print(f'Skipped {len(set(skipped_runs))} run(s) due to missing inputs.')

In [ ]:
summary_df = metrics_df.groupby('field', as_index=False).agg(
    rmse_mean=('rmse', 'mean'),
    rmse_median=('rmse', 'median'),
    mae_mean=('mae', 'mean'),
    mape_mean=('mape_pct_clipped', 'mean'),
    n_runs=('run_id', 'nunique'),
)
display(summary_df.sort_values('rmse_mean'))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)
axes = np.asarray(axes).ravel()

for ax, fname in zip(axes, target_fields):
    d = diag_df[(diag_df['field'] == fname) & (diag_df['true'] > 0) & (diag_df['pred'] > 0)]
    if len(d) == 0:
        ax.text(0.5, 0.5, 'No positive points', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(fname)
        continue
    ax.scatter(d['true'], d['pred'], s=3, alpha=0.08, rasterized=True)
    lo = float(min(d['true'].min(), d['pred'].min()))
    hi = float(max(d['true'].max(), d['pred'].max()))
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.1)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(f'True {fname}')
    ax.set_ylabel(f'Pred {fname}')
    ax.set_title(f'{fname}: test-set match')

for ax in axes[len(target_fields):]:
    ax.set_visible(False)

plt.show()

In [ ]:
def residual_vs_radius(profile_cache, field_name, q=(0.16, 0.5, 0.84)):
    ys_true = []
    ys_pred = []
    rb = None
    for rec in profile_cache:
        rb = rec['radial_bins']
        j = rec['field_names'].index(field_name)
        ys_true.append(rec['y_true'][:, :, j])
        ys_pred.append(rec['y_pred'][:, :, j])
    yt = np.vstack(ys_true)
    yp = np.vstack(ys_pred)
    resid = yp - yt
    qq = np.quantile(resid, q=np.asarray(q), axis=0)
    return rb, qq

fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True, sharex=True)
axes = np.asarray(axes).ravel()

for ax, fname in zip(axes, target_fields):
    r, qq = residual_vs_radius(profile_cache, fname)
    ax.plot(r, qq[1], lw=2, label='median residual')
    ax.fill_between(r, qq[0], qq[2], alpha=0.25, label='16-84%')
    ax.axhline(0.0, color='k', ls='--', lw=1)
    ax.set_xscale('log')
    ax.set_xlabel('radius [kpc]')
    ax.set_ylabel('pred - true')
    ax.set_title(fname)
    ax.legend(fontsize=8)

for ax in axes[len(target_fields):]:
    ax.set_visible(False)

plt.show()

In [ ]:
# 4x4 population-level redshift comparison with interpolation-based aggregation (artifact-resistant).

snap_to_z = {90: 0.0, 74: 0.5, 60: 1.0, 44: 2.0}
snap_list = [90, 74, 60, 44]
plot_fields = ['pressure', 'temperature', 'gas_density', 'metallicity']
field_label_map = {
    'pressure': 'pressure',
    'temperature': 'temperature',
    'gas_density': 'gas density',
    'metallicity': 'metallicity',
}
MAX_POPULATION_RUNS = 20
N_SAMPLES_POP = 20
N_R_COMMON = 80

# Prefer an emulator that contains all requested fields.
if all(f in set(emu.available_fields()) for f in plot_fields):
    emu_pop = emu
    emu_pop_label = str(RUN_DIR)
elif 'emu_z' in globals() and all(f in set(emu_z.available_fields()) for f in plot_fields):
    emu_pop = emu_z
    emu_pop_label = str(TEMPORAL_RUN_DIR) if 'TEMPORAL_RUN_DIR' in globals() else 'temporal_model'
else:
    raise ValueError(
        f"No loaded emulator has all requested fields {plot_fields}. "
        f"Available in emu: {emu.available_fields()}"
    )

def run_has_all_snaps(run_id: int, snaps):
    for s in snaps:
        try:
            _ = resolve_profile_file(
                run_id,
                base_path=Path(args.profiles_base),
                suite=args.suite,
                sim_set=args.sim_set,
                snapnum=int(s),
            )
        except FileNotFoundError:
            return False
    return True

theta_table_z = pd.read_csv(Path(args.param_csv))
theta_has_run_id = 'run_id' in theta_table_z.columns
if theta_has_run_id:
    theta_table_z = theta_table_z.set_index('run_id')

def get_theta_for_run(run_id: int):
    if theta_has_run_id:
        return theta_table_z.loc[int(run_id)].to_numpy(dtype=np.float32)
    return theta_table_z.iloc[int(run_id)].to_numpy(dtype=np.float32)

def _interp_log_profile(r_src, y_src, r_dst):
    r_src = np.asarray(r_src, dtype=np.float64).reshape(-1)
    y_src = np.asarray(y_src, dtype=np.float64).reshape(-1)
    m = np.isfinite(r_src) & np.isfinite(y_src) & (r_src > 0.0) & (y_src > 0.0)
    if np.count_nonzero(m) < 2:
        return np.full(r_dst.shape, np.nan, dtype=np.float64)
    xr = np.log10(r_src[m])
    yr = np.log10(y_src[m])
    order = np.argsort(xr)
    xr = xr[order]
    yr = yr[order]
    xdst = np.log10(r_dst)
    return np.interp(xdst, xr, yr, left=np.nan, right=np.nan)

candidate_runs = [int(rid) for rid in test_run_ids if run_has_all_snaps(int(rid), snap_list)]
if len(candidate_runs) == 0:
    raise RuntimeError('No test-set run has all requested snapshots (90,74,60,44).')

population_runs = candidate_runs[:MAX_POPULATION_RUNS]
print(f'Using emulator: {emu_pop_label}')
print(f'Using {len(population_runs)} runs for population comparison (of {len(candidate_runs)} available).')

# Store interpolated profiles per (field, snapshot) and run-level errors.
profiles_acc = {(f, int(s)): {'true': [], 'pred': []} for f in plot_fields for s in snap_list}
summary_rows = []
skipped_pairs = []

for snap in snap_list:
    run_records = []

    for run_sel in population_runs:
        try:
            fp = resolve_profile_file(
                run_sel,
                base_path=Path(args.profiles_base),
                suite=args.suite,
                sim_set=args.sim_set,
                snapnum=int(snap),
            )
            theta_sel = get_theta_for_run(run_sel)
        except (FileNotFoundError, KeyError, IndexError) as exc:
            skipped_pairs.append((int(run_sel), int(snap), str(exc)))
            continue

        with np.load(fp) as dat:
            m500c = dat['M500c'].astype(np.float64)
            r500c = dat['R500c'].astype(np.float64)
            radial_bins = dat['radial_bins'].astype(np.float64)
            y_true = np.stack([
                dat['pressure_array'].astype(np.float64),
                dat['temperature_array'].astype(np.float64),
                dat['gas_density_array'].astype(np.float64),
                dat['metallicity_array'].astype(np.float64),
            ], axis=-1)

        rr500 = (radial_bins[None, :] / np.maximum(r500c[:, None], 1e-12)).astype(np.float32)

        try:
            pred = emu_pop.predict(
                theta=theta_sel,
                M=m500c.astype(np.float32),
                r_bins=rr500,
                field=plot_fields,
                snapnum=int(snap),
                n_samples=N_SAMPLES_POP,
            )
        except TypeError:
            pred = emu_pop.predict(
                theta=theta_sel,
                M=m500c.astype(np.float32),
                r_bins=rr500,
                field=plot_fields,
                n_samples=N_SAMPLES_POP,
            )

        y_pred = pred.mean.astype(np.float64)
        r_phys = (rr500 * r500c[:, None].astype(np.float32)).astype(np.float64)

        run_records.append({
            'run_id': int(run_sel),
            'r_phys': r_phys,
            'y_true': y_true,
            'y_pred': y_pred,
            'n_halos': int(m500c.shape[0]),
        })

        for j, fname in enumerate(plot_fields):
            yt = y_true[:, :, j]
            yp = y_pred[:, :, j]
            m = np.isfinite(yt) & np.isfinite(yp) & (yt > 0.0) & (yp > 0.0)
            if np.any(m):
                dlog = np.log10(yp[m]) - np.log10(yt[m])
                rmse_log10 = float(np.sqrt(np.mean(dlog ** 2)))
            else:
                rmse_log10 = np.nan
            summary_rows.append({
                'run_id': int(run_sel),
                'snap': int(snap),
                'z': float(snap_to_z.get(int(snap), np.nan)),
                'field': fname,
                'n_halos': int(m500c.shape[0]),
                'rmse_log10': rmse_log10,
            })

    if len(run_records) == 0:
        continue

    # Define a common physical-radius grid over overlap to remove binning artifacts.
    rmin_overlap = max(float(np.nanmin(rec['r_phys'])) for rec in run_records)
    rmax_overlap = min(float(np.nanmax(rec['r_phys'])) for rec in run_records)
    if not np.isfinite(rmin_overlap) or not np.isfinite(rmax_overlap) or rmax_overlap <= rmin_overlap:
        continue
    r_common = np.geomspace(max(rmin_overlap, 1e-3), rmax_overlap, N_R_COMMON)

    for rec in run_records:
        r_phys = rec['r_phys']
        y_true = rec['y_true']
        y_pred = rec['y_pred']
        for h in range(r_phys.shape[0]):
            for j, fname in enumerate(plot_fields):
                yt_i = _interp_log_profile(r_phys[h, :], y_true[h, :, j], r_common)
                yp_i = _interp_log_profile(r_phys[h, :], y_pred[h, :, j], r_common)
                profiles_acc[(fname, int(snap))]['true'].append(yt_i)
                profiles_acc[(fname, int(snap))]['pred'].append(yp_i)

    for fname in plot_fields:
        profiles_acc[(fname, int(snap))]['r_common'] = r_common

# 4x4 figure: rows=fields, cols=redshifts.
fig, axes = plt.subplots(
    len(plot_fields),
    len(snap_list),
    figsize=(16, 12),
    constrained_layout=False,
    sharex='col',
    sharey='row',
)
axes = np.asarray(axes)
summary_df = pd.DataFrame(summary_rows)

true_color = 'tab:blue'
model_color = 'tab:orange'

for i, fname in enumerate(plot_fields):
    for j, snap in enumerate(snap_list):
        ax = axes[i, j]
        rec = profiles_acc.get((fname, int(snap)), None)
        if rec is None or len(rec['true']) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_xscale('log')
            continue

        r_common = rec['r_common']
        true_arr = np.asarray(rec['true'], dtype=np.float64)
        pred_arr = np.asarray(rec['pred'], dtype=np.float64)

        true_med = np.nanmedian(true_arr, axis=0)
        pred_med = np.nanmedian(pred_arr, axis=0)
        pred_q16 = np.nanquantile(pred_arr, 0.16, axis=0)
        pred_q84 = np.nanquantile(pred_arr, 0.84, axis=0)

        # Error on the median for true points (robust approximation).
        true_n = np.sum(np.isfinite(true_arr), axis=0).astype(np.float64)
        true_mad = np.nanmedian(np.abs(true_arr - true_med[None, :]), axis=0)
        true_sigma_rob = 1.4826 * true_mad
        true_med_err = 1.2533 * true_sigma_rob / np.sqrt(np.maximum(true_n, 1.0))

        valid = (
            np.isfinite(true_med)
            & np.isfinite(pred_med)
            & np.isfinite(pred_q16)
            & np.isfinite(pred_q84)
            & np.isfinite(true_med_err)
            & (true_n >= 3)
        )
        if np.count_nonzero(valid) == 0:
            ax.text(0.5, 0.5, 'No valid profiles', ha='center', va='center', transform=ax.transAxes)
            ax.set_xscale('log')
            continue

        # True data: points with error bars only (no line).
        valid_idx = np.where(valid)[0]
        step = max(1, len(valid_idx) // 18)
        plot_idx = valid_idx[::step]
        if len(plot_idx) == 0 or plot_idx[-1] != valid_idx[-1]:
            plot_idx = np.unique(np.append(plot_idx, valid_idx[-1]))
        ax.errorbar(
            r_common[plot_idx],
            true_med[plot_idx],
            yerr=true_med_err[plot_idx],
            fmt='o',
            ms=2.6,
            lw=0.9,
            elinewidth=0.8,
            capsize=1.8,
            color=true_color,
            alpha=0.95,
            label='true median ± err(median)',
        )

        # Model: line + same-color uncertainty contour.
        ax.plot(r_common[valid], pred_med[valid], lw=1.9, color=model_color, ls='--', label='pred median')
        ax.fill_between(
            r_common[valid],
            pred_q16[valid],
            pred_q84[valid],
            color=model_color,
            alpha=0.22,
            label='pred 16-84%',
        )
        ax.set_xscale('log')
        ax.set_title(f'snap={snap} (z={snap_to_z.get(int(snap), "n/a")})', fontsize=10)

        if j == 0:
            ax.set_ylabel(f"log10 {field_label_map.get(fname, fname)}")
        if i == len(plot_fields) - 1:
            ax.set_xlabel('radius [kpc]')

        d = summary_df[(summary_df['field'] == fname) & (summary_df['snap'] == int(snap))]
        if len(d) > 0:
            n_runs = int(d['run_id'].nunique())
            n_halos = int(d['n_halos'].sum())
            rmse_m = float(np.nanmean(d['rmse_log10']))
            ax.text(
                0.03, 0.03,
                f'runs={n_runs}\nhalos={n_halos:,}\nRMSE(log10)={rmse_m:.3f}',
                transform=ax.transAxes,
                fontsize=7,
                va='bottom',
                ha='left',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'),
            )

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.985))
fig.suptitle('Population redshift comparison (4x4): interpolation-based physical-radius aggregation', fontsize=14, y=0.998)
fig.tight_layout(rect=[0, 0, 1, 0.965])
plt.show()

population_redshift_df = summary_df.copy()
if not population_redshift_df.empty:
    pop_summary = population_redshift_df.groupby(['field', 'snap', 'z'], as_index=False).agg(
        n_runs=('run_id', 'nunique'),
        total_halos=('n_halos', 'sum'),
        rmse_log10_mean=('rmse_log10', 'mean'),
        rmse_log10_median=('rmse_log10', 'median'),
    ).sort_values(['field', 'z'])
    display(pop_summary)
else:
    print('No population summary could be computed (all run/snapshot pairs were skipped).')

if skipped_pairs:
    print(f'Skipped {len(skipped_pairs)} run/snapshot pair(s) due to missing inputs.')

In [ ]:
# Re-render single-simulation figure allowing very low halo counts (including single-halo snapshots).
if 'single_profiles' not in globals() or len(single_profiles) == 0:
    raise RuntimeError('Run the previous single-simulation cell first.')

fig, axes = plt.subplots(4, 4, figsize=(16, 12), constrained_layout=False, sharex='col', sharey='row')
axes = np.asarray(axes)

true_color = 'tab:blue'
model_color = 'tab:orange'

for i, fname in enumerate(single_fields):
    for j, snap in enumerate(single_snap_list):
        ax = axes[i, j]
        if int(snap) not in single_profiles:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_xscale('log')
            continue

        prof = single_profiles[int(snap)]
        r_common = prof['r_common']
        true_arr = prof['true_by_field'][fname]
        pred_arr = prof['pred_by_field'][fname]

        true_med = np.nanmedian(true_arr, axis=0)
        pred_med = np.nanmedian(pred_arr, axis=0)
        pred_q16 = np.nanquantile(pred_arr, 0.16, axis=0)
        pred_q84 = np.nanquantile(pred_arr, 0.84, axis=0)

        true_n = np.sum(np.isfinite(true_arr), axis=0).astype(np.float64)
        true_mad = np.nanmedian(np.abs(true_arr - true_med[None, :]), axis=0)
        true_sigma_rob = 1.4826 * true_mad
        true_med_err = 1.2533 * true_sigma_rob / np.sqrt(np.maximum(true_n, 1.0))
        true_med_err[~np.isfinite(true_med_err)] = 0.0

        valid = (
            np.isfinite(true_med)
            & np.isfinite(pred_med)
            & np.isfinite(pred_q16)
            & np.isfinite(pred_q84)
            & np.isfinite(true_med_err)
            & (true_n >= 1)
        )
        if np.count_nonzero(valid) == 0:
            ax.text(0.5, 0.5, 'No valid profiles', ha='center', va='center', transform=ax.transAxes)
            ax.set_xscale('log')
            continue

        valid_idx = np.where(valid)[0]
        step = max(1, len(valid_idx) // 18)
        plot_idx = valid_idx[::step]
        if len(plot_idx) == 0 or plot_idx[-1] != valid_idx[-1]:
            plot_idx = np.unique(np.append(plot_idx, valid_idx[-1]))

        ax.errorbar(
            r_common[plot_idx],
            true_med[plot_idx],
            yerr=true_med_err[plot_idx],
            fmt='o',
            ms=2.6,
            lw=0.9,
            elinewidth=0.8,
            capsize=1.8,
            color=true_color,
            alpha=0.95,
            label='true median ± err(median)',
        )
        ax.plot(r_common[valid], pred_med[valid], lw=1.9, color=model_color, ls='--', label='pred median')
        ax.fill_between(
            r_common[valid],
            pred_q16[valid],
            pred_q84[valid],
            color=model_color,
            alpha=0.22,
            label='pred 16-84%',
        )
        ax.set_xscale('log')
        ax.set_title(f'snap={snap} (z={snap_to_z.get(int(snap), "n/a")})', fontsize=10)

        if j == 0:
            ax.set_ylabel(f"log10 {single_field_label_map.get(fname, fname)}")
        if i == 3:
            ax.set_xlabel('radius [kpc]')

        d = single_summary_df[(single_summary_df['field'] == fname) & (single_summary_df['snap'] == int(snap))]
        if len(d) > 0:
            rmse_m = float(np.nanmean(d['rmse_log10']))
            n_halos = int(d['n_halos'].iloc[0])
            ax.text(
                0.03, 0.03,
                f'run={run_single}\nhalos={n_halos:,}\nRMSE(log10)={rmse_m:.3f}',
                transform=ax.transAxes,
                fontsize=7,
                va='bottom',
                ha='left',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'),
            )

handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.985))
fig.suptitle(f'Single test simulation (run_id={run_single}) across redshift: 4x4', fontsize=14, y=0.998)
fig.tight_layout(rect=[0, 0, 1, 0.965])
plt.show()

In [ ]:
import importlib
import inspect
import anp_emulator.api as anp_api

# Reload emulator API in case this kernel was started before code changes.
importlib.reload(anp_api)
EmulatorFresh = anp_api.Emulator

# Recreate emulator from run dir so predict signature is up to date.
emu_native = EmulatorFresh.from_run_dir(RUN_DIR, device=DEVICE)

radial_bins = np.logspace(-1, np.log10(3), 50, dtype=np.float32)  # r/R500
M500 = float(10**13)  # Msun
field_names = ['pressure', 'temperature', 'gas_density', 'metallicity']
z_grid = np.linspace(0.0, 2.0, 25)

if not all(f in set(emu_native.available_fields()) for f in field_names):
    raise ValueError(f'Required fields not available in emulator: {field_names}')

has_native_z = bool(getattr(emu_native, 'use_continuous_redshift_feature', False))
print(f'Checkpoint supports native continuous-z feature: {has_native_z}')
if not has_native_z:
    print('Warning: this checkpoint was not trained with continuous-z input; redshift values will map to nearest snapshot context only.')

supports_redshift_kw = 'redshift' in inspect.signature(emu_native.predict).parameters
if not supports_redshift_kw:
    raise RuntimeError(
        'Loaded Emulator.predict does not accept redshift=. Ensure anp_emulator/api.py is updated and rerun this cell.'
    )

# z=0 anchor prediction for ratio normalization.
pred_z0 = emu_native.predict(
    theta=fiducial_params,
    M=np.asarray([M500], dtype=np.float32),
    r_bins=radial_bins,
    field=field_names,
    redshift=0.0,
    n_samples=N_SAMPLES_EVOL,
).mean[0].astype(np.float64)

ratio_cube = []
for z in z_grid:
    pred_z = emu_native.predict(
        theta=fiducial_params,
        M=np.asarray([M500], dtype=np.float32),
        r_bins=radial_bins,
        field=field_names,
        redshift=float(z),
        n_samples=N_SAMPLES_EVOL,
    ).mean[0].astype(np.float64)
    ratio_cube.append(pred_z / np.clip(pred_z0, 1e-30, None))
ratio_cube = np.asarray(ratio_cube, dtype=np.float64)  # (n_z, n_r, n_fields)

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, constrained_layout=True)
axes = np.asarray(axes).ravel()
norm = plt.Normalize(vmin=float(z_grid.min()), vmax=float(z_grid.max()))
cmap = plt.cm.viridis

for j, fname in enumerate(field_names):
    ax = axes[j]
    for k, z in enumerate(z_grid):
        ratio = ratio_cube[k, :, j]
        valid = np.isfinite(ratio) & (ratio > 0.0)
        if np.count_nonzero(valid) >= 2:
            ax.plot(radial_bins[valid], ratio[valid]-1, color=cmap(norm(float(z))), lw=1.1, alpha=0.95)
    ax.axhline(0.0, color='k', ls='--', lw=1.0)
    ax.set_xscale('log')
    # ax.set_yscale('log')
    ax.set_title(fname)
    ax.set_xlabel('r / R500')
    ax.set_ylabel('pred(z) / pred(z=0)')

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.tolist(), shrink=0.96, pad=0.01)
cbar.set_label('redshift z (native API input)')

fig.suptitle(f'Native redshift-conditioned evolution at fixed M500={M500:.2e} Msun', fontsize=13)
plt.show()

print('Available model snapshots:', list(emu_native.snapnums))
if hasattr(emu_native, 'redshift_by_snap') and len(emu_native.redshift_by_snap) > 0:
    pairs = sorted([(float(z), int(s)) for s, z in emu_native.redshift_by_snap.items()], key=lambda t: t[0])
    print('Snapshot context nodes (z -> snap):', pairs)